# Day 3 - Conversational AI - aka Chatbot!

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyDJ


In [3]:
# Initialize

openai = OpenAI()
MODEL = 'gpt-4o-mini'

In [4]:
system_message = "You are a helpful assistant"

# Please read this! A change from the video:

In the video, I explain how we now need to write a function called:

`chat(message, history)`

Which expects to receive `history` in a particular format, which we need to map to the OpenAI format before we call OpenAI:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```

But Gradio has been upgraded! Now it will pass in `history` in the exact OpenAI format, perfect for us to send straight to OpenAI.

So our work just got easier!

We will write a function `chat(message, history)` where:  
**message** is the prompt to use  
**history** is the past conversation, in OpenAI format  

We will combine the system message, history and latest message, then call OpenAI.

In [ ]:
이 부분은 Gradio가 OpenAI 챗 API 포맷과 바로 호환되도록 업데이트되면서 생긴 변화예요. 각각의 역할을 정리해드릴게요.

1️⃣ message

현재 사용자가 입력한 프롬프트

즉, 지금 막 보낸 "user 메시지" 하나를 담고 있어요.

예시:

message = "Tell me a joke about cats"

2️⃣ history

이전 대화 전체를 저장한 기록

OpenAI 챗 API가 요구하는 포맷({"role": ..., "content": ...})으로 이미 들어와 있어요.

보통 user, assistant, system 역할이 번갈아 기록됨.

예시:

history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi there, how can I help you?"}
]

3️⃣ 합쳐서 쓰는 방식

chat(message, history) 함수 안에서는:

system message (어시스턴트의 기본 규칙)

history (이전 대화 맥락)

message (사용자가 방금 입력한 것)

이 세 가지를 합쳐서 OpenAI API 호출에 넘깁니다.

def chat(message, history):
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    messages.extend(history)           # 과거 대화 기록 추가
    messages.append({"role": "user", "content": message})  # 새 입력 추가
    
    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=messages
    )
    return response["choices"][0]["message"]["content"]


✅ 요약

message: 이번에 사용자가 입력한 문장.

history: 직전까지의 대화 맥락 (이미 OpenAI 형식).

두 개를 합쳐서 OpenAI API에 그대로 전달하면, 맥락 있는 대화를 이어갈 수 있음.

In [5]:
# Simpler than in my video - we can easily create this function that calls OpenAI
# It's now just 1 line of code to prepare the input to OpenAI!

# Student Octavio O. has pointed out that this isn't quite as straightforward for Claude -
# see the excellent contribution in community-contributions "Gradio_issue_with_Claude" that handles Claude.

def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    print("History is:")
    print(history)
    print("And messages is:")
    print(messages)

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
코드 해석
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]


messages 리스트를 구성하는 부분

system_message: 모델의 기본 규칙이나 성격을 알려주는 system role 메시지.

history: Gradio가 넘겨주는 이전 대화 기록 (이미 {"role": ..., "content": ...} 형식).

message: 방금 사용자가 입력한 새 user 메시지.

👉 이 세 가지를 합쳐 최종적으로 OpenAI API에 넘길 messages 리스트를 만든 거예요.

    print("History is:")
    print(history)
    print("And messages is:")
    print(messages)


디버깅용 출력문.

history와 최종 messages가 어떤 모양으로 API에 전달되는지 확인할 수 있음.

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)


OpenAI의 스트리밍 모드 호출.

model=MODEL: 사용할 모델 지정 (예: "gpt-4o-mini").

messages=messages: 위에서 만든 대화 기록 전체 전달.

stream=True: 답변이 한 번에 나오지 않고, 토큰 단위로 조금씩 흘러나오도록 설정. (실시간 UI 출력 가능)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response


실시간 응답 처리 부분

stream은 generator처럼 하나씩 chunk(토큰 단위 응답)를 반환.

chunk.choices[0].delta.content: 새로 도착한 문자열 조각.

response에 누적.

yield response: 중간 결과를 계속 반환 → Gradio UI에 실시간 업데이트됨.

👉 yield 덕분에 사용자는 모델이 다 쓰기 전에 글자가 차례차례 나타나는 걸 볼 수 있어요.

정리

message: 현재 사용자 입력.

history: 이전 대화 기록.

messages: system + history + message 합친 것.

stream=True와 yield를 사용해 실시간 스트리밍 출력 구현.

## And then enter Gradio's magic!

In [6]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


In [ ]:
type="messages" 의 역할

gr.ChatInterface는 Gradio에서 채팅 UI를 만들어주는 편리한 래퍼예요.
여기서 type 파라미터는 fn 함수(chat 함수)의 입력/출력 형식을 어떻게 다룰지를 결정합니다.

1. 기본값 (type="text")

fn(message, history)에서 message는 문자열(str), history는 튜플 리스트(list of tuples)로 들어와요.

즉, history = [("user가 말한 것", "assistant가 말한 것"), ...] 형식.

2. type="messages"

이 경우 OpenAI Chat API와 똑같은 포맷으로 history를 넘겨줍니다.

즉, history = [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}] 형식.

그래서 chat() 함수 안에서 별도의 변환 과정 없이 history를 바로 OpenAI API에 넘길 수 있는 거예요.

3. 정리

type="text" → 단순 텍스트 중심, history는 (user, assistant) 튜플.

type="messages" → OpenAI 포맷 그대로, history는 {"role": ..., "content": ...} 리스트.

따라서 지금 코드에서 type="messages"를 쓴 이유는,
chat() 함수가 OpenAI API의 messages 형식과 바로 호환되도록 하기 위함입니다.

👉 요약:
type="messages"는 Gradio가 history를 OpenAI ChatCompletion API와 호환되는 포맷으로 전달하도록 해 주는 옵션이에요.

혹시 제가 type="text"일 때와 type="messages"일 때, 같은 입력이 들어왔을 때 history가 어떻게 다르게 생기는지 예시를 코드로 보여드릴까요?

나의 말:
그럼 claude나 gemini를 쓸거면 type을 다른 걸 쓰거나 형태 변환해주는 코드를 써얗 ㅏ니
ChatGPT의 말:

맞습니다 ✅

핵심은 Gradio가 넘겨주는 history 포맷이 OpenAI 전용이라는 거예요.
그래서 Claude, Gemini처럼 메시지 포맷이 다른 API를 쓸 때는 두 가지 방법 중 하나를 택해야 합니다:

1️⃣ type="text"를 쓰고 직접 변환

type="text"로 두면 history가 (user, assistant) 튜플 리스트로 들어옵니다.

이걸 Claude나 Gemini가 요구하는 포맷으로 직접 변환하는 거죠.

예시 (Claude용):

def chat(message, history):
    # history: [("Hello", "Hi there!"), ("How are you?", "I'm good.")]
    
    messages = []
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": message})

    response = claude.messages.create(
        model="claude-3-haiku-20240307",
        system="You are a helpful assistant.",
        messages=messages,
    )
    return response.content[0].text

2️⃣ type="messages"를 쓰되 변환 코드 추가

그대로 OpenAI 포맷({"role": ..., "content": ...})을 받고,

Claude/Gemini가 원하는 방식으로 맵핑 변환을 해줍니다.

예시 (Claude용):

def chat(message, history):
    # history: [{"role": "user", "content": "Hello"}, {"role": "assistant", "content": "Hi"}]
    
    messages = []
    for msg in history:
        # Claude도 role=user/assistant를 지원하지만,
        # 일부 필드 이름이나 구조가 다를 수 있음 → 변환 필요
        messages.append({"role": msg["role"], "content": msg["content"]})
    messages.append({"role": "user", "content": message})

    response = claude.messages.create(
        model="claude-3-haiku-20240307",
        system="You are a helpful assistant.",
        messages=messages,
    )
    return response.content[0].text

3️⃣ Gemini의 경우

Gemini는 OpenAI 호환 endpoint를 제공할 때는 OpenAI 포맷 그대로 쓸 수 있어요.

하지만 Google 공식 SDK(google.generativeai)를 쓰면 contents 형식이라 변환 필요.

예시 (Gemini SDK용):

def chat(message, history):
    # OpenAI-style history를 Gemini-style contents로 변환
    contents = []
    for msg in history:
        role = "user" if msg["role"] == "user" else "model"
        contents.append({"role": role, "parts": [{"text": msg["content"]}]})
    contents.append({"role": "user", "parts": [{"text": message}]})

    response = gemini_model.generate_content(contents)
    return response.text


✅ 요약:

type="messages" → OpenAI 포맷 그대로 들어옴 → Claude/Gemini SDK에 맞게 변환 필요

type="text" → 단순 텍스트 튜플로 들어옴 → 직접 원하는 구조로 조립

👉 정리하면: Claude나 Gemini를 쓰려면

type="messages" 그대로 쓰고 변환 함수 만들거나,

**type="text"**로 받고 튜플 리스트 → 메시지 변환 해주면 돼요.

In [7]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [8]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [9]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [9]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [10]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


In [12]:
# Fixed a bug in this function brilliantly identified by student Gabor M.!
# I've also improved the structure of this function

def chat(message, history):

    relevant_system_message = system_message
    if 'belt' in message:
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
사용자가 벨트를 요청하면 
시스템 메시지 추가해

In [13]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [11]:
import time, random
# -*- coding: utf-8 -*-
import os, time, random
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

# ENV & client
load_dotenv(override=True)  # OPENAI_API_KEY 필요
client = OpenAI()

# --- Config ---
MODEL = "gpt-4o-mini"
STORE_POLICY = {
    "no_sell": ["belt", "belts"],
    "sale": {"hats": 0.60, "default": 0.50}
}
CATALOG = [
    {"sku":"HAT-001","name":"Classic Wool Hat","category":"hats","colors":["black","navy"],"sizes":["M","L"],"price":40000},
    {"sku":"HAT-002","name":"Canvas Bucket Hat","category":"hats","colors":["beige","olive"],"sizes":["Free"],"price":32000},
    {"sku":"TS-101","name":"Basic Tee","category":"tops","colors":["white","black"],"sizes":["S","M","L"],"price":22000},
    {"sku":"JK-201","name":"Light Windbreaker","category":"outer","colors":["navy"],"sizes":["M","L"],"price":69000},
]

# --- Helpers ---
def detect_intent_and_entities(text):
    lower = text.lower()
    intent = "ask"
    if any(k in lower for k in ["recommend","추천","what should","뭐가 좋","골라"]): intent = "recommend"
    elif any(k in lower for k in ["price","얼마","비싸","가격"]): intent = "price"
    elif any(k in lower for k in ["compare","비교","차이"]): intent = "compare"
    elif any(k in lower for k in ["buy","살","구매","장바구니"]): intent = "buy"

    category = None
    if any(k in lower for k in ["hat","hats","모자"]): category = "hats"
    elif any(k in lower for k in ["tee","t-shirt","tshirt","티","상의","top","tops"]): category = "tops"
    elif any(k in lower for k in ["jacket","아우터","자켓","outer"]): category = "outer"

    banned = any(k in lower for k in STORE_POLICY["no_sell"])
    unsure = any(k in lower for k in ["not sure","모르겠","애매","고민","추천해줘"])
    return {"intent": intent, "category": category, "banned": banned, "unsure": unsure}

def sale_price(item):
    rate = STORE_POLICY["sale"].get(item["category"], STORE_POLICY["sale"]["default"])
    return int(item["price"] * (1 - rate)), rate

def shortlist_catalog(category=None, limit=3):
    items = [x for x in CATALOG if (category is None or x["category"] == category)]
    return items[:limit]

def make_sales_copy(category_hint=None, unsure=False):
    intro = "좋은 선택이세요! 이번 주 세일이 아주 커요. 모자는 60% 할인, 그 외 대부분 품목은 50% 할인 중입니다."
    if category_hint == "hats" or unsure:
        intro += " 특히 모자 카테고리가 알찬 구성이에요."
    return intro

def system_prompt(user_flags):
    base = (
        "You are a helpful assistant for a clothes store.\n"
        "- Always prefer items from the provided CATALOG; do NOT invent items.\n"
        "- Gently encourage items on sale. Hats are 60% off; most others 50% off.\n"
        "- If an item is not sold (e.g., belts), say so and suggest on-sale alternatives.\n"
        "- Be concise, warm, and helpful. Offer 2-3 concrete options with final prices.\n"
        f"CATALOG (JSON): {CATALOG}\n"
    )
    if user_flags.get("banned"):
        base += "Reminder: The store does not sell belts. Offer hats or other on-sale items instead.\n"
    return base

def trim_history(messages, max_messages=8, max_chars=8000):
    trimmed = messages[-max_messages:]
    total, kept = 0, []
    for m in reversed(trimmed):
        total += len(m.get("content",""))
        kept.append(m)
        if total > max_chars: break
    return list(reversed(kept))

def build_user_visible_context(insight, picks):
    lines = [make_sales_copy(insight["category"], insight["unsure"])]
    if picks:
        lines.append("지금 보실만한 후보:")
        for it in picks:
            final, rate = sale_price(it)
            lines.append(f"- {it['name']} ({it['category']}) · {final}원 (↓{int(rate*100)}%)")
    return "\n".join(lines)

# --- Core chat (generator) ---
def chat(message, history, user_state=None, client=None):
    user_state = user_state or {"prefs": {}, "cart": []}
    insight = detect_intent_and_entities(message)
    category = "hats" if (insight["banned"] or insight["unsure"]) else insight["category"]
    candidates = shortlist_catalog(category, limit=3)
    sys_msg = system_prompt(insight)
    contextual_hint = build_user_visible_context(insight, candidates)

    messages = [{"role": "system", "content": sys_msg}]
    messages += trim_history(history)
    messages += [
        {"role": "assistant", "content": contextual_hint},
        {"role": "user", "content": message}
    ]

    max_retries, backoff = 3, 1.0
    for attempt in range(max_retries):
        try:
            stream = client.chat.completions.create(
                model=MODEL, messages=messages, stream=True, temperature=0.7
            )
            response = ""
            for chunk in stream:
                piece = getattr(chunk.choices[0].delta, "content", None) or ""
                response += piece
                yield response
            return
        except Exception:
            if attempt == max_retries - 1:
                yield "죄송합니다. 잠시 오류가 발생했습니다. 다시 한 번만 시도해 주세요. (네트워크/할당량 문제일 수 있어요)"
                return
            time.sleep(backoff + random.random() * 0.3)
            backoff *= 2

# --- history 포맷 자동 보정 (tuple → messages) ---
def _ensure_messages_format(history):
    # history가 [[user, assistant], ...] 형태면 messages로 변환
    if history and isinstance(history[0], list) and len(history[0]) == 2:
        msgs = []
        for u, a in history:
            if u: msgs.append({"role": "user", "content": u})
            if a: msgs.append({"role": "assistant", "content": a})
        return msgs
    return history or []

# --- Gradio wrapper ---
def chat_stream(message, history):
    history_msgs = _ensure_messages_format(history)
    for resp in chat(message, history_msgs, client=client):
        yield resp

# --- Launch (버튼 커스텀 파라미터 제거) ---
demo = gr.ChatInterface(
    fn=chat_stream,
    type="messages",
    title="🧢 Clothes Store Assistant (Sale Helper)",
    examples=[
        "벨트 있어요?",
        "모자 하나 추천해줘",
        "가벼운 아우터 있을까요? 예산 5만원대예요",
        "가격이랑 사이즈 알려주세요",
    ],
)

if __name__ == "__main__":
    demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7879
* Running on public URL: https://09782d2ba1bdaa9205.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Conversational Assistants are of course a hugely common use case for Gen AI, and the latest frontier models are remarkably good at nuanced conversation. And Gradio makes it easy to have a user interface. Another crucial skill we covered is how to use prompting to provide context, information and examples.
<br/><br/>
Consider how you could apply an AI Assistant to your business, and make yourself a prototype. Use the system prompt to give context on your business, and set the tone for the LLM.</span>
        </td>
    </tr>
</table>